**1.Cài thư viện**

In [1]:
pip install ipywidgets

Note: you may need to restart the kernel to use updated packages.


**2.Chạy giao diện đồ hoạ**

In [3]:

import joblib
import time
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML

# 1. Load model
model = None
try:
    model = joblib.load('sql_predictor.pkl')
    print("✅ Load model sql_predictor.pkl thành công!")
except Exception as e:
    print("❌ Lỗi load model:", e)
    print("👉 Hãy chạy notebook 02_ML_Model_Training.ipynb để tạo file sql_predictor.pkl trước.")

if model is None:
    raise FileNotFoundError("Không tìm thấy sql_predictor.pkl. Vui lòng train và lưu model trước khi chạy benchmark.")

FEATURE_INFO = {
    'has_like': ('LIKE Operator', 'Wildcard pattern scan', '0 (No LIKE)'),
    'has_group_by': ('GROUP BY Clause', 'Aggregation overhead', '0 (No GROUP BY)'),
    'has_order_by': ('ORDER BY Clause', 'Sorting overhead', '0 (No ORDER BY)'),
    'query_length': ('Query Length', 'SQL text length', '< 150 chars'),
    'has_where': ('WHERE Clause', 'Index filter condition', '1 (Filtering used)')
}

SLOW_THRESHOLD = 0.70
MODERATE_THRESHOLD = 0.40
IMPORTANT_THRESHOLD = 0.05

# 2. Hàm xử lý từng câu SQL
def analyze_single_sql(sql_query, query_index):
    sql_clean = sql_query.strip()
    if not sql_clean:
        return None, 0, "Low"

    query_upper = sql_clean.upper()
    feat_values = {
        'query_length': len(sql_clean),
        'has_where': 1 if 'WHERE' in query_upper else 0,
        'has_group_by': 1 if 'GROUP BY' in query_upper else 0,
        'has_order_by': 1 if 'ORDER BY' in query_upper else 0,
        'has_like': 1 if 'LIKE' in query_upper else 0
    }

    df_input = pd.DataFrame([feat_values])

    start_t = time.time()
    try:
        prob_slow = float(model.predict_proba(df_input)[0][1])
    except Exception:
        prob_slow = float(model.predict(df_input)[0])
    latency_ms = (time.time() - start_t) * 1000

    if prob_slow >= SLOW_THRESHOLD:
        risk_level = "High"
        badge_bg = "#ffebe9"
        badge_color = "#cf222e"
    elif prob_slow >= MODERATE_THRESHOLD:
        risk_level = "Moderate"
        badge_bg = "#fff8c5"
        badge_color = "#9a6700"
    else:
        risk_level = "Low"
        badge_bg = "#dafbe1"
        badge_color = "#1a7f37"

    confidence = "High" if abs(prob_slow - SLOW_THRESHOLD) > 0.20 else "Moderate"

    # Feature importance từ model
    try:
        importances = model.feature_importances_
        feature_names = ['query_length', 'has_where', 'has_group_by', 'has_order_by', 'has_like']
        feat_imp_map = dict(zip(feature_names, importances))
    except Exception:
        feat_imp_map = {
            'has_like': 0.451,
            'query_length': 0.295,
            'has_where': 0.252,
            'has_order_by': 0.001,
            'has_group_by': 0.0
        }

    sorted_features = sorted(feat_imp_map.items(), key=lambda x: x[1], reverse=True)

    table_rows_html = ""
    for f_name, imp_val in sorted_features:
        val = feat_values[f_name]
        imp_pct = round(float(imp_val) * 100, 1)
        f_title, f_desc, f_norm = FEATURE_INFO[f_name]
        
        # 1. Đánh giá Risk Badge dựa trên thực tế câu SQL
        if f_name == 'has_like':
            risk_badge = '<span style="background:#ffebe9; color:#cf222e; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">High Risk</span>' if val == 1 else '<span style="background:#f6f8fa; color:#57606a; padding:3px 8px; border-radius:4px; font-size:11px;">Optimal</span>'
        elif f_name in ['has_group_by', 'has_order_by']:
            risk_badge = '<span style="background:#fff8c5; color:#9a6700; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">Moderate Risk</span>' if val == 1 else '<span style="background:#f6f8fa; color:#57606a; padding:3px 8px; border-radius:4px; font-size:11px;">Optimal</span>'
        elif f_name == 'has_where':
            risk_badge = '<span style="background:#dafbe1; color:#1a7f37; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">Good Filter</span>' if val == 1 else '<span style="background:#ffebe9; color:#cf222e; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">No Filter</span>'
        elif f_name == 'query_length':
            if val > 300:
                risk_badge = '<span style="background:#ffebe9; color:#cf222e; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">High Risk</span>'
            elif val > 150:
                risk_badge = '<span style="background:#fff8c5; color:#9a6700; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">Moderate Risk</span>'
            else:
                risk_badge = '<span style="background:#dafbe1; color:#1a7f37; padding:3px 8px; border-radius:4px; font-weight:bold; font-size:11px;">Optimal</span>'
        else:
            risk_badge = '<span style="background:#f6f8fa; color:#57606a; padding:3px 8px; border-radius:4px; font-size:11px;">Optimal</span>'


        # 2. Xử lý thanh Bar cho cột Importance
        if imp_pct > 0:
            bar_html = f"""
            <div style="display:inline-flex; align-items:center; gap:8px; justify-content:center;">
                <div style="background:#e1e4e8; width:70px; height:7px; border-radius:4px; overflow:hidden; flex-shrink: 0;">
                    <div style="background:#0969da; width:{imp_pct}%; height:100%; border-radius:4px;"></div>
                </div>
                <span style="font-size:11px; font-weight:bold; color:#24292f;">{imp_pct}%</span>
            </div>"""
        else:
            bar_html = f"""
            <div style="display:inline-flex; align-items:center; gap:8px; justify-content:center;">
                <div style="background:#f0f2f5; width:70px; height:7px; border-radius:4px; flex-shrink: 0;"></div>
                <span style="font-size:11px; color:#8c959f;">0.0%</span>
            </div>"""

        # 3. Căn giữa cột Importance
        table_rows_html += f"""
        <tr style="border-bottom: 1px solid #e1e4e8; font-size: 12px;">
            <td style="padding: 10px; width: 25%;"><b>{f_title}</b><br><span style="color:#6e7781; font-size:10px;">{f_desc}</span></td>
            <td style="padding: 10px; text-align: center; width: 10%;"><b>{val}</b></td>
            <td style="padding: 10px; width: 20%;">{risk_badge}</td>
            <td style="padding: 10px; text-align: center; width: 28%;">{bar_html}</td>
            <td style="padding: 10px; color:#57606a; width: 17%; font-size:11px;">{f_norm}</td>
        </tr>
        """

    card_html = f"""
    <div style="background: white; border: 1px solid #e2e8f0; border-radius: 8px; padding: 16px; margin-bottom: 16px; box-shadow: 0 1px 3px rgba(0,0,0,0.05);">
        <div style="font-weight: bold; color: #1e293b; font-size: 13px; margin-bottom: 8px;">
            Query #{query_index}: <code style="background: #f1f5f9; padding: 3px 8px; border-radius: 4px; color: #0f172a;">{sql_clean}</code>
        </div>

        <div style="display: flex; gap: 20px; background: #f8fafc; padding: 10px 14px; border-radius: 6px; margin-bottom: 12px; font-size: 12px; flex-wrap: wrap;">
            <div>Status: <span style="background:{badge_bg}; color:{badge_color}; padding:2px 8px; border-radius:8px; font-weight:bold;">{risk_level}</span></div>
            <div>Slow Probability: <b>{prob_slow*100:.1f}%</b></div>
            <div>Confidence: <b>{confidence}</b></div>
            <div>Latency: <b>{latency_ms:.2f} ms</b></div>
        </div>

        <table style="width: 100%; border-collapse: collapse; background: white; table-layout: fixed;">
            <thead>
                <tr style="background: #f8fafc; border-bottom: 2px solid #e2e8f0; text-align: left; font-size: 11px; color: #64748b;">
                    <th style="padding: 8px 10px; width: 25%;">Feature</th>
                    <th style="padding: 8px 10px; text-align: center; width: 10%;">Value</th>
                    <th style="padding: 8px 10px; width: 20%;">Risk Assessment</th>
                    <th style="padding: 8px 10px; text-align: center; width: 28%;">Importance</th>
                    <th style="padding: 8px 10px; width: 17%;">Optimal Range</th>
                </tr>
            </thead>
            <tbody>
                {table_rows_html}
            </tbody>
        </table>
    </div>
    """
    return card_html, latency_ms, risk_level

# 3. Hàm tổng hợp Batch Analysis Dashboard
def generate_batch_dashboard(raw_sql_text):
    raw_queries = [q.strip() for q in raw_sql_text.split(';') if q.strip()]

    if not raw_queries:
        return "<p style='color:red;'>⚠️ Vui lòng nhập ít nhất một câu lệnh SQL!</p>"

    total_queries = len(raw_queries)
    high_count = 0
    moderate_count = 0
    total_latency = 0
    cards_html = ""

    for idx, q in enumerate(raw_queries, 1):
        card_code, lat, risk = analyze_single_sql(q, idx)
        if card_code:
            cards_html += card_code
            total_latency += lat
            if risk == "High":
                high_count += 1
            elif risk == "Moderate":
                moderate_count += 1

    avg_latency = total_latency / total_queries if total_queries > 0 else 0
    low_count = total_queries - high_count - moderate_count

    summary_header = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-width: 1000px; border: 1px solid #d0d7de; border-radius: 8px; overflow: hidden; margin-top: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.08);">
        <div style="background-color: #2563eb; color: white; padding: 14px 20px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 10px;">
            <div>
                <h3 style="margin:0; font-size:17px;">Multi-SQL Query Batch Assessment</h3>
                <span style="font-size:12px; opacity:0.9;">Analyzing {total_queries} database queries in parallel</span>
            </div>
            <span style="background:rgba(255,255,255,0.2); font-size:11px; padding:4px 10px; border-radius:12px;">MariaDB Real-time</span>
        </div>

        <div style="padding: 18px; background-color: #f8fafc; max-height: 550px; overflow-y: auto;">
            <div style="background: white; border: 1px solid #e2e8f0; border-radius: 8px; padding: 16px; margin-bottom: 18px; display: flex; justify-content: space-around; text-align: center; flex-wrap: wrap; gap: 12px;">
                <div>
                    <div style="font-size: 11px; color: #64748b;">TOTAL QUERIES</div>
                    <div style="font-size: 22px; font-weight: bold; color: #0f172a;">{total_queries}</div>
                </div>
                <div style="border-left: 1px solid #e2e8f0;"></div>
                <div>
                    <div style="font-size: 11px; color: #1a7f37;">LOW / OK</div>
                    <div style="font-size: 22px; font-weight: bold; color: #1a7f37;">{low_count}</div>
                </div>
                <div style="border-left: 1px solid #e2e8f0;"></div>
                <div>
                    <div style="font-size: 11px; color: #9a6700;">MODERATE</div>
                    <div style="font-size: 22px; font-weight: bold; color: #9a6700;">{moderate_count}</div>
                </div>
                <div style="border-left: 1px solid #e2e8f0;"></div>
                <div>
                    <div style="font-size: 11px; color: #cf222e;">HIGH</div>
                    <div style="font-size: 22px; font-weight: bold; color: #cf222e;">{high_count}</div>
                </div>
                <div style="border-left: 1px solid #e2e8f0;"></div>
                <div>
                    <div style="font-size: 11px; color: #2563eb;">AVG LATENCY</div>
                    <div style="font-size: 22px; font-weight: bold; color: #2563eb;">{avg_latency:.2f} ms</div>
                </div>
            </div>

            {cards_html}

            <div style="font-size: 11px; color: #64748b; margin-top: 10px;">
                💡 <b>Batch Assessment Tip:</b> Separate multiple queries with semicolons (<code>;</code>) to perform bulk optimization analysis.
            </div>
        </div>
    </div>
    """
    return summary_header

# 4. Widgets UI
default_multi_sql = """SELECT * FROM customer_transactions WHERE age > 30;
SELECT customer_name FROM customer_transactions WHERE customer_name LIKE '%Smith%' ORDER BY transaction_amount DESC;
SELECT status, COUNT(*) FROM customer_transactions GROUP BY status;"""

input_box = widgets.Textarea(
    value=default_multi_sql,
    placeholder="Enter multiple SQL queries separated by semicolons (';')...",
    layout=widgets.Layout(width='820px', height='90px')
)

send_btn = widgets.Button(
    description='Analyze Batch',
    button_style='primary',
    icon='layer-group',
    layout=widgets.Layout(height='90px', width='160px')
)

display_area = widgets.HTML(value=generate_batch_dashboard(input_box.value))

def on_click_analyze(b):
    display_area.value = generate_batch_dashboard(input_box.value)

send_btn.on_click(on_click_analyze)

# Hiển thị
display(display_area)
display(widgets.HBox([input_box, send_btn]))

✅ Load model sql_predictor.pkl thành công!


HTML(value='\n    <div style="font-family: -apple-system, BlinkMacSystemFont, \'Segoe UI\', Roboto, sans-serif…